# Pipeline ambx — Testes por Etapa

Este notebook executa e visualiza cada etapa da pipeline do framework **ambx** (*Ambient Access*), permitindo inspecionar os resultados intermediários antes de integrar tudo no script `pipeline.py`.

**Etapas:**
1. **1a — Malha Territorial**: geração da grade de análise (hexagonal/quadrada)
2. **1b — Pontos de Interesse**: coleta de POIs categorizados do OSM
3. **1c — Rede Viária**: construção do grafo de deslocamento
4. **1d — Vinculação Malha ↔ Rede**: snapping dos centróides aos nós do grafo
5. **2 — Roteamento A\*** : matriz origem-destino com caminhos mínimos (A* com heurística euclidiana)

In [1]:
# --- Setup: path e imports ---
import sys
sys.path.insert(0, "../scripts")


from ambx.grid import generate_grid, GridFormat
from ambx.network import (
    add_travel_time,
    get_network,
    get_graph_edges,
    project_network,
    snap_grid_to_network,
)
from ambx.pois import get_pois
from ambx.routing import snap_pois_to_network, routing_matrix

print("✓ Todos os módulos importados com sucesso.")

✓ Todos os módulos importados com sucesso.


## Parâmetros

Defina aqui a localidade e os parâmetros da análise. Altere `LOCATION` para testar com outras cidades.

In [2]:
# --- Parâmetros da análise ---
LOCATION = "Porto Alegre, Rio Grande do Sul, Brazil"
GRID_FORMAT = GridFormat.HEXAGON
CELL_SIZE = 200        # metros (raio do hexágono ou lado do quadrado)
POI_BUFFER = 2000      # metros — captura serviços de cidades vizinhas
NETWORK_TYPE = "walk"  # walk, bike, drive
WALK_SPEED_KPH = 5.0   # km/h

print(f"Localidade : {LOCATION}")
print(f"Malha      : {GRID_FORMAT.value}, {CELL_SIZE}m")
print(f"Buffer POI : {POI_BUFFER}m")
print(f"Rede       : {NETWORK_TYPE} ({WALK_SPEED_KPH} km/h)")

Localidade : Porto Alegre, Rio Grande do Sul, Brazil
Malha      : hexagon, 200m
Buffer POI : 2000m
Rede       : walk (5.0 km/h)


## Etapa 1a — Malha Territorial

Geração da grade de células (hexágonos ou quadrados) que cobre o contorno administrativo da localidade.

In [3]:
# --- Gerar malha territorial ---
grid = generate_grid(LOCATION, grid_format=GRID_FORMAT, cell_size=CELL_SIZE)
print(f"Células geradas: {len(grid)}")
print(f"CRS: {grid.crs}")
print(f"Área total coberta: {grid.to_crs(grid.estimate_utm_crs()).area.sum() / 1e6:.2f} km²")

Células geradas: 5064
CRS: EPSG:4326
Área total coberta: 526.27 km²


## Etapa 1b — Pontos de Interesse

Coleta de destinos do OpenStreetMap categorizados em saúde, educação, transporte e alimentação.

In [4]:
# --- Coletar Pontos de Interesse ---
pois = get_pois(LOCATION, buffer=POI_BUFFER)
print(f"Total de POIs: {len(pois)}")
print(f"Categorias:\n{pois['category'].value_counts().to_string()}")

Total de POIs: 1764
Categorias:
category
education         713
food              502
health            477
transportation     72


## Etapa 1c — Rede Viária

Construção do grafo de deslocamento a partir do OpenStreetMap, projeção para CRS métrico e cálculo do tempo de percurso nas arestas.

In [5]:
# --- Construir grafo viário ---
print("Baixando grafo do OSM...")
graph = get_network(LOCATION, network_type=NETWORK_TYPE)
print(f"Nós (original): {graph.number_of_nodes()}")
print(f"Arestas (original): {graph.number_of_edges()}")

# Projetar para UTM
print("Projetando para UTM...")
graph = project_network(graph)
print(f"CRS do grafo projetado: {graph.graph.get('crs', 'N/A')}")

# Adicionar tempo de percurso
graph = add_travel_time(graph, speed_kph=WALK_SPEED_KPH)
print(f"Tempo de percurso adicionado (speed={WALK_SPEED_KPH} km/h)")

# Visualizar arestas
edges = get_graph_edges(graph)
print(f"Extensão total da rede: {edges.length.sum() / 1000:.1f} km")

Baixando grafo do OSM...
Nós (original): 41907
Arestas (original): 113542
Projetando para UTM...
CRS do grafo projetado: EPSG:32722
Tempo de percurso adicionado (speed=5.0 km/h)
Extensão total da rede: 8439.9 km


## Etapa 1d — Vinculação Malha ↔ Rede

Snapping dos centróides de cada célula da malha ao nó mais próximo do grafo viário. Células muito distantes da rede são descartadas.

In [6]:
# --- Vincular malha à rede ---
snapped = snap_grid_to_network(grid, graph, projected=False, max_distance=1000)

print(f"Células originais      : {len(grid)}")
print(f"Células vinculadas     : {len(snapped)}")
print(f"Taxa de vinculação     : {len(snapped) / len(grid) * 100:.1f}%")
print(f"Distância média snap   : {snapped['snap_distance'].mean():.1f} m")
print(f"Distância máxima snap  : {snapped['snap_distance'].max():.1f} m")

# Células não vinculadas (se houver limite de distância)
if len(snapped) < len(grid):
    lost = len(grid) - len(snapped)
    print(f"Células descartadas    : {lost} ({lost / len(grid) * 100:.1f}%)")

Células originais      : 5064
Células vinculadas     : 4357
Taxa de vinculação     : 86.0%
Distância média snap   : 172.8 m
Distância máxima snap  : 997.9 m
Células descartadas    : 707 (14.0%)


## Etapa 2 — Roteamento A*

Cálculo da matriz origem-destino: para cada célula da malha, encontra o caminho mínimo até cada POI usando o algoritmo **A\*** com heurística de distância euclidiana. O custo das arestas é o tempo de percurso (`travel_time`).

In [7]:
# --- Roteamento A*: K vizinhos mais próximos por categoria ---
import time

K_NEAREST = 3   # quantos POIs mais próximos por categoria considerar
N_JOBS = 38

print("Vinculando POIs à rede...")
pois_snapped = snap_pois_to_network(pois, graph)
print(f"POIs vinculados: {len(pois_snapped)} / {len(pois)}")
print(f"Distância média snap: {pois_snapped['snap_distance'].mean():.1f} m")

# Estatísticas de quantos pares serão calculados
cats = pois_snapped["category"].nunique()
est_pairs = len(snapped) * cats * K_NEAREST
print(f"\nEstimativa de pares A*: {len(snapped)} células × {cats} categorias × {K_NEAREST} = {est_pairs}")
print(f"(matriz completa seria: {len(snapped)} × {len(pois_snapped)} = {len(snapped) * len(pois_snapped)})")
print()

t0 = time.perf_counter()
# snapped_sample = snapped.sample(n=10, random_state=42)  # amostra para teste
matrix = routing_matrix(snapped, pois_snapped, graph, k_nearest=K_NEAREST, speed_kph=WALK_SPEED_KPH, n_jobs=N_JOBS)
elapsed = time.perf_counter() - t0

print(f"\n✓ Matriz concluída em {elapsed:.1f}s ({len(matrix) / elapsed:.0f} pares/s)")

# --- Resumo da matriz ---
reachable = matrix["travel_time"].notna().sum()
unreachable = matrix["travel_time"].isna().sum()
print(f"\nPares alcançáveis:   {reachable} ({reachable / len(matrix) * 100:.1f}%)")
print(f"Pares inalcançáveis: {unreachable} ({unreachable / len(matrix) * 100:.1f}%)")

print(f"\nTempo de viagem (alcançáveis):")
tt = matrix.loc[matrix["travel_time"].notna(), "travel_time"]
print(f"  Média:   {tt.mean() / 60:.1f} min")
print(f"  Mediana: {tt.median() / 60:.1f} min")
print(f"  Mínimo:  {tt.min() / 60:.1f} min")
print(f"  Máximo:  {tt.max() / 60:.1f} min")

print(f"\nDistância euclidiana vs tempo de viagem:")
print(f"  Correlação: {matrix['euclidean_dist'].corr(matrix['travel_time']):.3f}")

Vinculando POIs à rede...
POIs vinculados: 1764 / 1764
Distância média snap: 233.0 m

Estimativa de pares A*: 4357 células × 4 categorias × 3 = 52284
(matriz completa seria: 4357 × 1764 = 7685748)

  [seleção] célula 1/4357, 12 pares acumulados
  [seleção] célula 50/4357, 600 pares acumulados
  [seleção] célula 100/4357, 1200 pares acumulados
  [seleção] célula 150/4357, 1800 pares acumulados
  [seleção] célula 200/4357, 2400 pares acumulados
  [seleção] célula 250/4357, 3000 pares acumulados
  [seleção] célula 300/4357, 3600 pares acumulados
  [seleção] célula 350/4357, 4200 pares acumulados
  [seleção] célula 400/4357, 4800 pares acumulados
  [seleção] célula 450/4357, 5400 pares acumulados
  [seleção] célula 500/4357, 6000 pares acumulados
  [seleção] célula 550/4357, 6600 pares acumulados
  [seleção] célula 600/4357, 7200 pares acumulados
  [seleção] célula 650/4357, 7800 pares acumulados
  [seleção] célula 700/4357, 8400 pares acumulados
  [seleção] célula 750/4357, 9000 pares acu

## Mapa das Células Sample — Tempo Médio de Acesso

Visualização espacial das células amostradas, coloridas pelo tempo médio de acesso (média dos tempos de viagem A* para os POIs mais próximos de cada categoria).

In [8]:
# Calcular tempo médio de acesso por célula
avg_travel_time = (
    matrix.groupby("cell_idx")["travel_time"]
    .mean()
    .reset_index()
    .rename(columns={"travel_time": "avg_travel_time"})
)

# Pegar geometrias das células sample (do grid original)
# snapped_sample tem o mesmo index do grid original
cells = grid.loc[snapped.index].copy()
cells["cell_idx"] = cells.index

# Juntar com os tempos médios
cells_plot = cells.merge(avg_travel_time, on="cell_idx", how="left")

cells_plot.explore(column="avg_travel_time", cmap="YlOrRd", legend=True)

## Resumo da Modelagem Territorial

Ao final destas 5 etapas, temos todos os elementos necessários para a análise de acessibilidade:

| Elemento | Descrição | Status |
|----------|-----------|:------:|
| **Malha** | Grade de células de análise | ✅ |
| **POIs** | Destinos categorizados (saúde, educação, transporte, alimentação) | ✅ |
| **Rede** | Grafo viário com tempos de percurso | ✅ |
| **Snapping** | Mapeamento célula → nó da rede | ✅ |
| **Routing (A\*)** | Matriz origem-destino com caminhos mínimos | ✅ |

**Próximas etapas:**
- **`environment`** — carregamento de camadas ambientais (rasters, polígonos de inundação)
- **`penalties`** — penalização das arestas por condicionantes ambientais
- **`indicators`** — cálculo de PTh, Índice G e F15

In [ ]:
# --- Carregar camada ambiental: inundação CPRM ---
import importlib

import ambx
from ambx.environment import build_environment
importlib.reload(ambx.environment)

# Área de interesse a partir da malha
area_of_interest = grid

# Carregar camada de inundação
environment_layers = build_environment(
    area_of_interest=area_of_interest,
    vector_paths=[
        "../data/raw/porto_alegre/inundacao_cprm.geojson",
        "../data/raw/porto_alegre/movimento_massa_cprm.geojson"
    ],
)

for layer in environment_layers.vectors:
    print(f"Camada vetorial: {layer.name}")
    print(f"CRS: {layer.gdf.crs}")
    print(f"Feições: {len(layer.gdf)}")
    print(f"Colunas: {list(layer.gdf.columns)}")
    print(f"\nValores únicos de classe: {layer.gdf['classe'].unique()}\n")

Camada vetorial: inundacao_cprm
CRS: EPSG:32722
Feições: 2199
Colunas: ['objectid', 'geometria', 'municipio', 'uf', 'processo', 'classe', 'obs', 'fonte', 'Shape__Area', 'Shape__Length', 'geometry']

Valores únicos de classe: <ArrowStringArray>
['Médio', 'Alto', 'Baixa']
Length: 3, dtype: str

Camada vetorial: movimento_massa_cprm
CRS: EPSG:32722
Feições: 302
Colunas: ['objectid', 'geomteria', 'municipio', 'uf', 'processo', 'classe', 'fonte', 'Shape__Area', 'Shape__Length', 'geometry']

Valores únicos de classe: <ArrowStringArray>
['Média', 'Baixa', 'Alta']
Length: 3, dtype: str



<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [22]:
print("Grid CRS:", grid.crs)
print("POIs CRS:", pois.crs)
print("Graph CRS:", graph.graph.get('crs', 'N/A'))
print(f"CRS da camada ambiental: {environment_layers.vectors[0].gdf.crs}")

Grid CRS: EPSG:4326
POIs CRS: EPSG:4326
Graph CRS: EPSG:32722
CRS da camada ambiental: EPSG:32722
